# exp01: Model Tabular dan Ensemble

Menguji algoritma klasik, kelompok sensor, preprocessing, seleksi fitur, balancing, interaksi, hierarchy, dan ensemble. Setiap window 15/25/45 detik diproses terpisah. Notebook mencakup training sampai evaluasi test dan penyimpanan model.

Notebook ini mandiri: tidak memerlukan exp02, shared, script Python, atau file requirements.


## Lokasi data

Dataset dibaca langsung dari Google Drive. Path awal sudah diisi untuk folder proyek tim. Biarkan RESUME_DIR = None untuk run baru. Untuk melanjutkan, isi dengan path Run yang sebelumnya dicetak, lalu jalankan kembali dari atas. Tidak ada mode smoke/full.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Penelitian-Adaptive-IDE")
DATASET_ROOT = PROJECT_ROOT / "Dataset" / "v2_session_phase"
RESUME_DIR = None
EXPERIMENT = "exp01"

## Environment

Cell ini memasang dependency yang diperlukan. Jika diminta restart, restart session dan jalankan kembali dari awal. Mount Drive menggunakan akun yang mempunyai izin Editor. Pintasan folder proyek di My Drive dapat digunakan. Environment harus cocok untuk resume; PyTorch/CUDA bawaan tidak diganti.


In [ ]:
import importlib.metadata as _metadata
import importlib.util as _import_util
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Gunakan runtime Colab dengan Python 3.12 atau lebih baru.")

# Versi core sama pada kedua notebook; dependency dikelola di sini, tanpa file txt.
_required = {
    "numpy": "2.5.1",
    "pandas": "3.0.3",
    "scipy": "1.18.0",
    "scikit-learn": "1.9.1",
    "matplotlib": "3.11.0",
    "cloudpickle": "3.1.3",
}
if EXPERIMENT == "exp01":
    _required.update(
        {"imbalanced-learn": "0.14.2", "xgboost": "3.4.1", "catboost": "1.2.10"}
    )
_imports = {"scikit-learn": "sklearn", "imbalanced-learn": "imblearn"}
_changes = []
_restart = False
_current: str | None
for _name, _version in _required.items():
    try:
        _current = _metadata.version(_name)
    except _metadata.PackageNotFoundError:
        _current = None
    if _current != _version:
        _changes.append(f"{_name}=={_version}")
        _restart = _restart or _imports.get(_name, _name) in sys.modules
if _changes:
    print("Menyiapkan dependency:", ", ".join(_changes))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *_changes]
    )  # nosec B603
    if _restart:
        raise RuntimeError(
            "Dependency diperbarui. Pilih Runtime > Restart session, lalu Run all kembali."
        )
if EXPERIMENT == "exp02" and _import_util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch tidak tersedia. Gunakan runtime Colab standar yang sudah menyertakan PyTorch."
    )

if _import_util.find_spec("google.colab") is not None:
    from google.colab import drive

    drive.mount("/content/drive")

print("Dependency siap. Path proyek:", PROJECT_ROOT)

In [ ]:
import copy
import hashlib
import io
import json
import platform
import random
import tempfile
import time
from datetime import UTC, datetime
from importlib import metadata
from pathlib import Path
from typing import Any

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    PolynomialFeatures,
    RobustScaler,
    StandardScaler,
)
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight

LABELS = ["Low", "Medium", "High"]
SEED = 42
CODE_VERSION = "batch03-1.0.0"
SENSOR_FEATURES = [
    "rmssd_ms",
    "sdnn_ms",
    "rmssd_baseline_ratio",
    "blink_count",
    "blink_rate_per_minute",
    "ear_left_mean",
    "ear_left_std",
    "ear_right_mean",
    "ear_right_std",
    "iris_size_estimate_mean",
    "iris_size_estimate_std",
    "iris_size_delta_mean",
    "iris_size_delta_std",
    "head_pitch_mean",
    "head_pitch_std",
    "head_yaw_mean",
    "head_yaw_std",
    "gaze_x_norm_mean",
    "gaze_x_norm_std",
    "gaze_y_norm_mean",
    "gaze_y_norm_std",
    "eye_state_fixation_ratio",
    "eye_state_saccade_ratio",
    "eye_state_transition_ratio",
    "eye_state_unclassified_ratio",
    "eye_state_other_ratio",
]
EYE_FEATURES = [
    f
    for f in SENSOR_FEATURES
    if f
    not in {
        "rmssd_ms",
        "sdnn_ms",
        "rmssd_baseline_ratio",
        "blink_count",
        "eye_state_other_ratio",
    }
]
GROUPS = {
    "eye": EYE_FEATURES,
    "hrv": ["rmssd_ms", "sdnn_ms"],
    "fusion": ["rmssd_ms", "sdnn_ms"] + EYE_FEATURES,
}


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)


def digest(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()


def metrics(y, probabilities):
    predicted = np.asarray(probabilities).argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y, predicted)),
        "macro_f1": float(
            f1_score(y, predicted, labels=[0, 1, 2], average="macro", zero_division=0)
        ),
        "balanced_accuracy": float(balanced_accuracy_score(y, predicted)),
    }


def combine_probabilities(arrays, weights):
    values = [np.asarray(a, dtype=float) for a in arrays]
    weights = np.asarray(weights, dtype=float)
    if (
        not values
        or len(values) != len(weights)
        or any(a.shape != values[0].shape for a in values)
    ):
        raise ValueError("Bentuk probabilitas atau jumlah bobot tidak cocok.")
    if values[0].ndim != 2 or values[0].shape[1] != 3 or not np.isfinite(weights).all():
        raise ValueError("Probabilitas harus mempunyai tiga kolom kelas.")
    if (weights < 0).any() or weights.sum() <= 0:
        raise ValueError("Bobot ensemble tidak valid.")
    if any(
        not np.isfinite(a).all()
        or (a < 0).any()
        or not np.allclose(a.sum(axis=1), 1.0, atol=1e-5)
        for a in values
    ):
        raise ValueError("Probabilitas tidak valid.")
    return np.average(np.stack(values), axis=0, weights=weights)


def rank_key(record):
    score = record["validation"]
    return (
        -score["accuracy"],
        -score["macro_f1"],
        record.get("complexity", 1),
        record["id"],
    )


def dump_bundle(bundle):
    return cloudpickle.dumps(bundle)


def load_bundle(payload):
    # Muat hanya artefak yang dibuat tim sendiri, bukan file dari sumber tak dikenal.
    return cloudpickle.loads(payload)  # nosec B301


IMPLEMENTATION_HASH = "6e43118bffe5a65db434f27628607d7c931b02c151d06b5a724807162a790e97"

try:
    from IPython.display import display
except ImportError:
    display = print

## Pembagian training, validation, dan test

Blok waktu 120 detik dibagi dengan seed 42 dan keterwakilan kelas dominan blok, mendekati 70/15/15. Data orang yang sama boleh muncul pada ketiga bagian. Ini evaluasi blok campuran, bukan LOPO atau TimeSeriesSplit.

Semua pilihan window memakai penempatan endpoint yang sama. Rentang pendukung 45 detik dipakai untuk purge: test dipertahankan, validation yang overlap test dibuang, lalu training yang overlap validation/test dibuang. Proporsi baris setelah purge dapat berbeda dari proporsi blok. Timestamp dan identitas hanya untuk pembagian data, bukan fitur model. Label langsung berasal dari cognitive_load_label.


In [ ]:
def prepare_views(frames):
    """Split metadata-only; tidak menggunakan nilai fitur untuk penempatan blok."""
    if set(frames) != {15, 25, 45}:
        raise ValueError("Ketiga CSV window 15, 25, dan 45 detik harus tersedia.")
    clean: dict[int, Any] = {}
    union: dict[tuple[str, int], str] = {}
    anchors: dict[str, int] = {}
    people: dict[str, str] = {}
    required = set(
        SENSOR_FEATURES
        + [
            "participant_id",
            "session_id",
            "window_start",
            "window_end",
            "window_seconds",
            "cognitive_load_label",
        ]
    )
    for window in (15, 25, 45):
        df = frames[window].copy()
        missing = required - set(df)
        if missing or df.empty:
            raise ValueError(
                f"Window {window}: kolom hilang {sorted(missing)} atau CSV kosong."
            )
        for field in ("session_id", "participant_id"):
            if df[field].isna().any() or df[field].astype(str).str.strip().eq("").any():
                raise ValueError(f"Window {window}: {field} kosong.")
            df[field] = df[field].astype(str)
        if not df["cognitive_load_label"].isin(LABELS).all():
            raise ValueError(f"Window {window}: label tidak dikenal atau kosong.")
        try:
            start = pd.to_datetime(
                df["window_start"], utc=True, errors="raise", format="ISO8601"
            )
            end = pd.to_datetime(
                df["window_end"], utc=True, errors="raise", format="ISO8601"
            )
            df["_start_ns"] = start.dt.as_unit("ns").astype("int64")
            df["_end_ns"] = end.dt.as_unit("ns").astype("int64")
        except (ValueError, TypeError) as exc:
            raise ValueError(f"Window {window}: timestamp tidak valid.") from exc
        if start.isna().any() or end.isna().any():
            raise ValueError("Timestamp kosong.")
        duration = (end - start).dt.total_seconds().to_numpy()
        if not np.allclose(duration, window, atol=0.001, rtol=0):
            raise ValueError(
                f"Window {window}: rentang timestamp bukan {window} detik."
            )
        if not pd.to_numeric(df["window_seconds"], errors="coerce").eq(window).all():
            raise ValueError(f"Window {window}: window_seconds tidak cocok.")
        if df.duplicated(["session_id", "_end_ns"]).any():
            raise ValueError(f"Window {window}: endpoint ganda.")
        for field in SENSOR_FEATURES:
            df[field] = pd.to_numeric(df[field], errors="coerce").replace(
                [np.inf, -np.inf], np.nan
            )
        df["_y"] = df["cognitive_load_label"].map(
            {label: i for i, label in enumerate(LABELS)}
        )
        df = df.sort_values(["session_id", "_end_ns"]).reset_index(drop=True)
        for row in df[
            [
                "session_id",
                "participant_id",
                "_start_ns",
                "_end_ns",
                "cognitive_load_label",
            ]
        ].itertuples(index=False, name=None):
            sid, person, first, last, label = row
            key = (sid, int(last))
            if key in union and union[key] != label:
                raise ValueError("Label endpoint bertentangan antarwindow.")
            if sid in people and people[sid] != person:
                raise ValueError("Satu session_id terhubung ke beberapa peserta.")
            people[sid] = person
            union[key] = label
            anchors[sid] = min(anchors.get(sid, int(first)), int(first))
        clean[window] = df

    blocks: dict[tuple[str, int], list[tuple[tuple[str, int], str]]] = {}
    for key, label in sorted(union.items()):
        block = (key[0], (key[1] - anchors[key[0]]) // 120_000_000_000)
        blocks.setdefault(block, []).append((key, label))
    buckets: dict[str, list[tuple[str, int]]] = {}
    for block, members in sorted(blocks.items()):
        counts: dict[str, int] = {}
        for _, label in members:
            counts[label] = counts.get(label, 0) + 1
        dominant = sorted(counts, key=lambda label: (-counts[label], label))[0]
        buckets.setdefault(dominant, []).append(block)
    rng = random.Random(SEED)
    assignment = {}
    for label, group in sorted(buckets.items()):
        rng.shuffle(group)
        n = max(1, round(len(group) * 0.15))
        if len(group) < 3:
            raise ValueError("Blok terlalu sedikit untuk membentuk tiga bagian.")
        for i, block in enumerate(group):
            part = "test" if i < n else "validation" if i < 2 * n else "train"
            for key, _ in blocks[block]:
                assignment[key] = part

    def close_to(key, index):
        values = index.get(key[0], np.array([], dtype=np.int64))
        if not len(values):
            return False
        pos = np.searchsorted(values, key[1])
        return any(
            abs(int(values[j]) - key[1]) < 45_000_000_000
            for j in (pos - 1, pos)
            if 0 <= j < len(values)
        )

    def index_for(parts):
        index: dict[str, list[int]] = {}
        for sid, last in parts:
            index.setdefault(sid, []).append(last)
        return {
            sid: np.array(sorted(values), dtype=np.int64)
            for sid, values in index.items()
        }

    kept = {key: part for key, part in assignment.items() if part == "test"}
    test_index = index_for(kept)
    kept.update(
        {
            key: part
            for key, part in assignment.items()
            if part == "validation" and not close_to(key, test_index)
        }
    )
    held_index = index_for(kept)
    kept.update(
        {
            key: part
            for key, part in assignment.items()
            if part == "train" and not close_to(key, held_index)
        }
    )
    manifest = [
        {
            "session_id": key[0],
            "end_ns": key[1],
            "assigned": assignment[key],
            "split": kept.get(key, "purged"),
        }
        for key in sorted(union)
    ]
    summary = []
    for window, df in clean.items():
        df["_split"] = [
            kept.get((sid, int(last)), "purged")
            for sid, last in zip(df["session_id"], df["_end_ns"], strict=True)
        ]
        for part in ("train", "validation", "test", "purged"):
            subset = df[df["_split"] == part]
            class_counts = subset["cognitive_load_label"].value_counts()
            if part != "purged" and not set(LABELS).issubset(class_counts.index):
                raise ValueError(
                    f"Window {window}: {part} tidak mempunyai ketiga kelas."
                )
            summary.append(
                {
                    "window_seconds": window,
                    "split": part,
                    "rows": len(subset),
                    **{label: int(class_counts.get(label, 0)) for label in LABELS},
                }
            )
    return clean, manifest, pd.DataFrame(summary)


def read_datasets(dataset_root):
    frames, hashes = {}, {}
    for window in (15, 25, 45):
        path = Path(dataset_root) / f"window_{window}s_step_5s" / "good_acceptable.csv"
        payload = path.read_bytes()
        hashes[str(window)] = hashlib.sha256(payload).hexdigest()
        frames[window] = pd.read_csv(io.BytesIO(payload))
    views, manifest, summary = prepare_views(frames)
    return views, manifest, summary, hashes


def part_frame(view, part):
    return view.loc[view["_split"] == part].reset_index(drop=True)


def feature_names(recipe):
    result = list(GROUPS[recipe["group"]])
    if recipe.get("ratio", False) and recipe["group"] != "eye":
        result.append("rmssd_baseline_ratio")
    return result


class SafeSelector(TransformerMixin, BaseEstimator):
    def __init__(self, selector="all"):
        self.selector = selector

    def fit(self, x, y=None):
        self.estimator_ = None
        if self.selector in ("mi8", "mi16"):
            k = min(int(self.selector[2:]), x.shape[1])

            def information(a, b):
                return mutual_info_classif(a, b, random_state=SEED)

            self.estimator_ = SelectKBest(information, k=k).fit(x, y)
        elif self.selector == "pca":
            if x.shape[1] > 1 and np.any(np.var(x, axis=0) > 1e-15):
                self.estimator_ = PCA(n_components=0.95, svd_solver="full").fit(x)
        elif self.selector != "all":
            raise ValueError("Selector tidak dikenal.")
        return self

    def transform(self, x):
        return x if self.estimator_ is None else self.estimator_.transform(x)


def make_preprocessor(columns, recipe):
    log_columns = [
        i
        for i, col in enumerate(columns)
        if recipe.get("log", False) and col in {"rmssd_ms", "sdnn_ms"}
    ]
    steps = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))]
    if log_columns:
        steps.append(
            (
                "log",
                ColumnTransformer(
                    [("hrv", FunctionTransformer(np.log1p), log_columns)],
                    remainder="passthrough",
                    verbose_feature_names_out=False,
                ),
            )
        )
    if recipe.get("scaler", "standard") != "none" or recipe.get("selector") == "pca":
        scaler = (
            RobustScaler() if recipe.get("scaler") == "robust" else StandardScaler()
        )
        steps.append(("scaler", scaler))
    if recipe.get("polynomial", False):
        steps += [
            (
                "interactions",
                PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
            ),
            ("interaction_scaler", StandardScaler()),
        ]
    steps.append(("selector", SafeSelector(recipe.get("selector", "all"))))
    return Pipeline(steps)


def sensor_input(frame, columns):
    x = frame[columns].copy().replace([np.inf, -np.inf], np.nan)
    for col in ("rmssd_ms", "sdnn_ms", "rmssd_baseline_ratio"):
        if col in x:
            x.loc[x[col] < 0, col] = np.nan
    return x

In [ ]:
views, split_manifest, split_summary, dataset_hashes = read_datasets(DATASET_ROOT)
display(split_summary)
missingness = pd.DataFrame(
    [
        {
            "window": window,
            "feature": feature,
            "missing_fraction": float(view[feature].isna().mean()),
        }
        for window, view in views.items()
        for feature in SENSOR_FEATURES
    ]
)
display(missingness[missingness["missing_fraction"] > 0].round(4))
print("Split SHA-256:", digest(split_manifest))
print("Target 70/15/15 berdasarkan blok; jumlah baris berubah setelah purge.")

## Penyimpanan dan resume

Setiap kandidat yang selesai diverifikasi lalu disimpan ke Drive. Runtime baru dapat melanjutkan kandidat selesai tanpa training ulang. Kandidat yang terputus sebelum selesai diulang dari awal; tidak ada janji resume di tengah epoch. Dataset, split, kode, dan versi environment harus cocok. Checkpoint rusak menghasilkan error, bukan dianggap selesai.


In [ ]:
class ArtifactStore:
    """Checkpoint lokal; salinan Drive diverifikasi sebelum completion marker."""

    def __init__(self, local, remote, signature):
        self.local, self.remote = Path(local), Path(remote)
        self.signature = signature
        self.local.mkdir(parents=True, exist_ok=True)
        self.remote.mkdir(parents=True, exist_ok=True)
        identity = self.remote / "run.json"
        expected = {"signature": signature}
        if identity.exists():
            if json.loads(identity.read_text(encoding="utf-8")) != expected:
                raise ValueError(
                    "Resume ditolak: dataset, split, konfigurasi, atau environment berubah."
                )
        else:
            self._copy_verified(identity, canonical(expected).encode())

    def _copy_verified(self, target, payload):
        target.parent.mkdir(parents=True, exist_ok=True)
        for attempt in range(3):
            try:
                # Tidak mengandalkan atomic rename pada mount Google Drive.
                with target.open("wb") as handle:
                    handle.write(payload)
                    handle.flush()
                if (
                    hashlib.sha256(target.read_bytes()).digest()
                    != hashlib.sha256(payload).digest()
                ):
                    raise OSError(f"Checksum salinan tidak cocok: {target}")
                return
            except OSError:
                if attempt == 2:
                    raise
                time.sleep(0.2 * (attempt + 1))

    def _folder(self, key):
        if not key or any(
            c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-"
            for c in key
        ):
            raise ValueError("ID artefak tidak valid.")
        return self.remote / key

    def commit(self, key, record, artifacts=None):
        folder = self._folder(key)
        artifacts = dict(artifacts or {})
        artifacts["record.json"] = canonical(record).encode()
        checksums = {}
        for name, payload in artifacts.items():
            if Path(name).name != name or name in {"done.json", "run.json"}:
                raise ValueError("Nama artefak tidak valid.")
            target = self.local / key / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)
            checksums[name] = hashlib.sha256(payload).hexdigest()
            self._copy_verified(folder / name, payload)
        self._copy_verified(
            folder / "done.json", canonical({"checksums": checksums}).encode()
        )

    def read(self, key):
        folder = self._folder(key)
        marker = folder / "done.json"
        if not marker.exists():
            return None
        try:
            done = json.loads(marker.read_text(encoding="utf-8"))
            for name, checksum in done["checksums"].items():
                if (
                    Path(name).name != name
                    or hashlib.sha256((folder / name).read_bytes()).hexdigest()
                    != checksum
                ):
                    raise ValueError(f"Artefak rusak/tidak lengkap: {key}/{name}")
            return json.loads((folder / "record.json").read_text(encoding="utf-8"))
        except (OSError, KeyError, json.JSONDecodeError) as exc:
            raise ValueError(f"Checkpoint tidak dapat diverifikasi: {key}") from exc

    def artifact(self, key, name):
        if self.read(key) is None:
            raise ValueError(f"Checkpoint belum selesai: {key}")
        if Path(name).name != name:
            raise ValueError("Nama artefak tidak valid.")
        return (self._folder(key) / name).read_bytes()

    def publish(self, name, payload):
        if Path(name).name != name:
            raise ValueError("Nama output tidak valid.")
        self._copy_verified(self.remote / name, payload)


def array_bytes(value):
    stream = io.BytesIO()
    np.save(stream, np.asarray(value), allow_pickle=False)
    return stream.getvalue()


def from_array_bytes(payload):
    return np.load(io.BytesIO(payload), allow_pickle=False)


def figure_bytes(figure):
    stream = io.BytesIO()
    figure.savefig(stream, format="png", dpi=150, bbox_inches="tight")
    plt.close(figure)
    return stream.getvalue()


def environment_versions(experiment):
    names = ["numpy", "pandas", "scipy", "scikit-learn", "cloudpickle", "matplotlib"]
    names += (
        ["imbalanced-learn", "xgboost", "catboost"]
        if experiment == "exp01"
        else ["torch"]
    )
    return {name: metadata.version(name) for name in names}


def start_run(project_root, experiment, hashes, manifest, resume_dir=None):
    signature = {
        "code_version": CODE_VERSION,
        "experiment": experiment,
        "dataset_sha256": hashes,
        "split_sha256": digest(manifest),
        "environment": environment_versions(experiment),
        "python": platform.python_version(),
        "implementation": IMPLEMENTATION_HASH,
    }
    if resume_dir is None:
        run_id = (
            datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ") + "_" + digest(signature)[:8]
        )
        remote = (
            Path(project_root) / "models" / "batch-03" / "outputs" / experiment / run_id
        )
    else:
        remote = Path(resume_dir)
        expected_parent = (
            Path(project_root) / "models" / "batch-03" / "outputs" / experiment
        )
        if (
            remote.parent.resolve() != expected_parent.resolve()
            or not (remote / "run.json").is_file()
        ):
            raise ValueError(
                "RESUME_DIR harus menunjuk run yang sudah ada pada eksperimen ini."
            )
    local = Path(tempfile.mkdtemp(prefix=f"batch03-{experiment}-"))
    store = ArtifactStore(local, remote, signature)
    store.publish("split.json", canonical(manifest).encode())
    print(f"Run: {remote}")
    print("Simpan path Run ini jika perlu melanjutkan runtime yang terputus.")
    return store

In [ ]:
def final_evaluation(store, views, finalists, refit, predict):
    """Pemenang dikunci dengan validation sebelum satu pun metrik test dihitung."""
    primary = sorted(finalists, key=rank_key)[0]
    selection = {
        "primary_id": primary["id"],
        "primary_window": primary["window"],
        "criterion": "validation_accuracy_then_macro_f1_then_complexity",
        "finalists": finalists,
    }
    previous = store.read("selection")
    if previous is not None and previous != selection:
        raise ValueError(
            "Finalist berubah pada run yang sudah dikunci; gunakan run baru."
        )
    if previous is None:
        store.commit("selection", selection)
    rows = []
    for candidate in finalists:
        window = candidate["window"]
        key = "final_" + candidate["id"]
        record = store.read(key)
        if record is None:
            view = views[window]
            trainval = view.loc[
                view["_split"].isin(["train", "validation"])
            ].reset_index(drop=True)
            test = part_frame(view, "test")
            started = time.perf_counter()
            model = refit(candidate, trainval)
            probabilities = predict(model, test)
            payload = dump_bundle(model)
            restored = load_bundle(payload)
            np.testing.assert_allclose(
                probabilities, predict(restored, test), atol=1e-6, rtol=1e-5
            )
            scores = metrics(test["_y"], probabilities)
            matrix = confusion_matrix(
                test["_y"], probabilities.argmax(axis=1), labels=[0, 1, 2]
            )
            report = classification_report(
                test["_y"],
                probabilities.argmax(axis=1),
                labels=[0, 1, 2],
                target_names=LABELS,
                output_dict=True,
                zero_division=0,
            )
            record = {
                "id": key,
                "candidate_id": candidate["id"],
                "window": window,
                "primary": candidate["id"] == primary["id"],
                "test_rows": len(test),
                "training_rows": len(trainval),
                "validation": candidate["validation"],
                "test": scores,
                "confusion_matrix": matrix.tolist(),
                "classification_report": report,
                "seconds": time.perf_counter() - started,
                "evaluated_model": "model.pkl",
            }
            predictions = test[
                [
                    "participant_id",
                    "session_id",
                    "window_start",
                    "window_end",
                    "cognitive_load_label",
                ]
            ].copy()
            predictions["predicted_label"] = [
                LABELS[i] for i in probabilities.argmax(axis=1)
            ]
            for i, label in enumerate(LABELS):
                predictions[f"probability_{label}"] = probabilities[:, i]
            columns = sorted(
                set(
                    col
                    for item in flatten_models(model)
                    for col in item.get("columns", [])
                )
            )
            fig, ax = plt.subplots(figsize=(5, 4))
            ax.imshow(matrix, cmap="Blues")
            for (i, j), count in np.ndenumerate(matrix):
                ax.text(
                    j,
                    i,
                    str(count),
                    ha="center",
                    va="center",
                    color="white" if count > matrix.max() / 2 else "black",
                )
            ax.set(
                xticks=range(3),
                yticks=range(3),
                xticklabels=LABELS,
                yticklabels=LABELS,
                xlabel="Prediksi",
                ylabel="Label dataset",
                title=f"Test — window {window}s",
            )
            store.commit(
                key,
                record,
                {
                    "model.pkl": payload,
                    "predictions.csv": predictions.to_csv(index=False).encode(),
                    "classification_report.csv": pd.DataFrame(report)
                    .T.to_csv()
                    .encode(),
                    "confusion_matrix.png": figure_bytes(fig),
                    "feature_schema.json": canonical(
                        {
                            "raw_sensor_features": columns,
                            "labels": LABELS,
                            "recipe": candidate["recipe"],
                        }
                    ).encode(),
                },
            )
        rows.append(record)
        print(
            f"Window {window}s — test accuracy {record['test']['accuracy']:.2%}, "
            f"Macro-F1 {record['test']['macro_f1']:.2%}; N={record['test_rows']}"
        )
        print(pd.DataFrame(record["classification_report"]).T.round(4).to_string())
        fig, ax = plt.subplots(figsize=(5, 4))
        matrix = np.asarray(record["confusion_matrix"])
        ax.imshow(matrix, cmap="Blues")
        for (i, j), value in np.ndenumerate(matrix):
            ax.text(j, i, str(value), ha="center", va="center")
        ax.set(
            xticks=range(3),
            yticks=range(3),
            xticklabels=LABELS,
            yticklabels=LABELS,
            xlabel="Prediksi",
            ylabel="Label dataset",
            title=f"Test — window {window}s",
        )
        plt.show()
    summary = pd.DataFrame(
        [
            {
                "window": r["window"],
                "primary": r["primary"],
                "test_rows": r["test_rows"],
                **r["test"],
            }
            for r in rows
        ]
    )
    store.publish("test_summary.csv", summary.to_csv(index=False).encode())
    chosen = next(r for r in rows if r["primary"])
    text = (
        "# Hasil run batch-03\n\n"
        f"Model utama dipilih berdasarkan validation: window {chosen['window']} detik.\n\n"
        f"- Accuracy test: {chosen['test']['accuracy']:.2%}\n"
        f"- Macro-F1 test: {chosen['test']['macro_f1']:.2%}\n"
        f"- Balanced accuracy test: {chosen['test']['balanced_accuracy']:.2%}\n"
        f"- Jumlah test: {chosen['test_rows']}\n"
        f"- Target accuracy 75%: {'tercapai' if chosen['test']['accuracy'] >= 0.75 else 'belum tercapai'}\n\n"
        f"Model evaluasi: {chosen['id']}/model.pkl. Model ini di-fit pada training + validation, "
        "tidak pada test. Window lain merupakan pembanding yang telah ditetapkan sebelum evaluasi.\n\n"
        "Split memakai blok waktu campuran dengan purge; data orang yang sama dapat muncul "
        "di training dan test. Hasil ini merupakan evaluasi internal dataset saat ini.\n"
    )
    store.publish("RESULTS.md", text.encode())
    store.commit(
        "complete",
        {"primary": chosen["id"], "test_summary": summary.to_dict(orient="records")},
    )
    print(text)
    return summary


def flatten_models(model):
    if model["kind"] == "ensemble":
        return [part for child in model["models"] for part in flatten_models(child)]
    if model["kind"] == "hierarchy":
        return flatten_models(model["first"]) + flatten_models(model["second"])
    return [model]


def publish_leaderboard(store, table):
    store.publish("validation_leaderboard.csv", table.to_csv(index=False).encode())


def plot_validation(table):
    best = table.loc[table.groupby(["window", "model"])["accuracy"].idxmax()]
    pivot = best.pivot(index="model", columns="window", values="accuracy")
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot.bar(ax=ax)
    ax.set(ylabel="Accuracy validation", xlabel="", ylim=(0, 1))
    ax.legend(title="Window (detik)")
    plt.tight_layout()
    plt.show()

In [ ]:
store = start_run(PROJECT_ROOT, EXPERIMENT, dataset_hashes, split_manifest, RESUME_DIR)
store.publish("split_counts.csv", split_summary.to_csv(index=False).encode())
store.publish("missingness.csv", missingness.to_csv(index=False).encode())

## Pipeline model

Delapan algoritma dibandingkan pada eye-only, HRV-only, dan fusion. Input bersih mengeluarkan blink_count dan eye_state_other_ratio. Rasio baseline HRV ditambahkan sebagai kandidat, bukan fitur wajib. Nama peserta, fase, difficulty, kualitas, dan timestamp tidak menjadi predictor.

Imputasi, transformasi, scaler, dan seleksi fitur hanya dipelajari dari training. ROS dan SMOTE hanya menambah training, bukan CSV sumber, validation, atau test. Probabilitas selalu memakai urutan Low, Medium, High.

SVM memakai kalibrasi sigmoid tiga bagian di dalam training untuk menghasilkan probabilitas. Validation dan test tidak dipakai untuk kalibrasi. Ini mengikuti API scikit-learn 1.9, bukan penggunaan SVC(probability=True) yang deprecated.


In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler


def base_recipe(family, group):
    return {
        "kind": "single",
        "family": family,
        "group": group,
        "ratio": False,
        "log": False,
        "selector": "all",
        "polynomial": False,
        "balance": "none",
        "scaler": "standard" if family in {"logistic", "svm", "knn"} else "none",
        "params": {},
        "seed": SEED,
    }


def make_estimator(family, params, seed, nclasses=3):
    p = dict(params)
    if family == "dummy":
        return DummyClassifier(strategy="most_frequent")
    if family == "logistic":
        return LogisticRegression(C=p.pop("C", 1.0), max_iter=3000, **p)
    if family == "svm":
        return CalibratedClassifierCV(
            SVC(random_state=seed, **p),
            method="sigmoid",
            cv=3,
            ensemble=False,
            n_jobs=1,
        )
    if family == "knn":
        return KNeighborsClassifier(n_jobs=2, **p)
    if family in {"random_forest", "extra_trees"}:
        cls = (
            RandomForestClassifier
            if family == "random_forest"
            else ExtraTreesClassifier
        )
        return cls(n_estimators=300, random_state=seed, n_jobs=2, **p)
    if family == "hist_gradient":
        return HistGradientBoostingClassifier(
            max_iter=p.pop("max_iter", 200),
            early_stopping=False,
            random_state=seed,
            **p,
        )
    if family == "xgboost":
        from xgboost import XGBClassifier

        return XGBClassifier(
            n_estimators=p.pop("n_estimators", 300),
            max_depth=p.pop("max_depth", 3),
            learning_rate=p.pop("learning_rate", 0.03),
            objective="multi:softprob" if nclasses == 3 else "binary:logistic",
            eval_metric="mlogloss" if nclasses == 3 else "logloss",
            tree_method="hist",
            n_jobs=2,
            random_state=seed,
            **p,
        )
    if family == "catboost":
        from catboost import CatBoostClassifier

        return CatBoostClassifier(
            iterations=p.pop("iterations", 300),
            depth=p.pop("depth", 4),
            learning_rate=p.pop("learning_rate", 0.03),
            loss_function="MultiClass" if nclasses == 3 else "Logloss",
            verbose=False,
            allow_writing_files=False,
            thread_count=2,
            random_seed=seed,
            **p,
        )
    raise ValueError(f"Algoritma tidak dikenal: {family}")


def fitted_single(recipe, frame, y):
    columns = feature_names(recipe)
    preprocessor = make_preprocessor(columns, recipe)
    values = preprocessor.fit_transform(sensor_input(frame, columns), y)
    if not np.isfinite(values).all():
        raise ValueError("Preprocessing menghasilkan nilai nonfinite.")
    balance = recipe["balance"]
    if balance == "ros":
        values, y = RandomOverSampler(random_state=recipe["seed"]).fit_resample(
            values, y
        )
    elif balance == "smote":
        count = min(np.bincount(y))
        if count < 2:
            raise ValueError("Training tidak cukup untuk SMOTE.")
        values, y = SMOTE(
            k_neighbors=min(5, count - 1), random_state=recipe["seed"]
        ).fit_resample(values, y)
    estimator = make_estimator(
        recipe["family"], recipe["params"], recipe["seed"], len(np.unique(y))
    )
    fit_args = {}
    if balance == "weight":
        if recipe["family"] == "knn":
            raise ValueError("KNN tidak mendukung class/sample weight saat fitting.")
        fit_args["sample_weight"] = compute_sample_weight("balanced", y)
    estimator.fit(values, y, **fit_args)
    return {
        "kind": "single",
        "recipe": recipe,
        "columns": columns,
        "preprocessor": preprocessor,
        "estimator": estimator,
    }


def fit_tabular(recipe, frame, y):
    y = np.asarray(y, dtype=int)
    if recipe["kind"] == "ensemble":
        return {
            "kind": "ensemble",
            "recipe": recipe,
            "models": [fit_tabular(child, frame, y) for child in recipe["children"]],
            "weights": recipe["weights"],
        }
    if recipe["kind"] == "hierarchy":
        child = recipe["child"]
        first = fitted_single(child, frame, (y != 0).astype(int))
        elevated = y != 0
        second = fitted_single(child, frame.loc[elevated], y[elevated] - 1)
        return {"kind": "hierarchy", "recipe": recipe, "first": first, "second": second}
    return fitted_single(recipe, frame, y)


def raw_probabilities(model, frame):
    transformed = model["preprocessor"].transform(sensor_input(frame, model["columns"]))
    return model["estimator"].predict_proba(transformed)


def predict_tabular(model, frame):
    if model["kind"] == "ensemble":
        return combine_probabilities(
            [predict_tabular(child, frame) for child in model["models"]],
            model["weights"],
        )
    if model["kind"] == "hierarchy":
        elevated = raw_probabilities(model["first"], frame)[:, 1]
        upper = raw_probabilities(model["second"], frame)
        return np.column_stack(
            [1 - elevated, elevated * upper[:, 0], elevated * upper[:, 1]]
        )
    probabilities = raw_probabilities(model, frame)
    aligned = np.zeros((len(frame), 3))
    for i, label in enumerate(model["estimator"].classes_):
        aligned[:, int(label)] = probabilities[:, i]
    return aligned


def evaluate_candidate(store, window, recipe, train, validation, stage):
    key = f"w{window}_" + digest(recipe)[:20]
    previous = store.read(key)
    if previous is not None:
        return previous
    started = time.perf_counter()
    model = fit_tabular(recipe, train, train["_y"].to_numpy())
    train_p = predict_tabular(model, train)
    val_p = predict_tabular(model, validation)
    record = {
        "id": key,
        "window": window,
        "recipe": recipe,
        "stage": stage,
        "complexity": 2 if recipe["kind"] in {"ensemble", "hierarchy"} else 1,
        "validation": metrics(validation["_y"], val_p),
        "training": metrics(train["_y"], train_p),
        "seconds": time.perf_counter() - started,
    }
    store.commit(
        key,
        record,
        {"model.pkl": dump_bundle(model), "validation.npy": array_bytes(val_p)},
    )
    print(
        f"{key} / {stage}: validation accuracy={record['validation']['accuracy']:.4f}, "
        f"Macro-F1={record['validation']['macro_f1']:.4f}"
    )
    return record


def tune_recipes(recipe, count=12):
    spaces: dict[str, dict[str, list[Any]]] = {
        "logistic": {"C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]},
        "svm": {
            "C": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
            "gamma": [0.0001, 0.001, 0.01, 0.1, 1.0],
        },
        "knn": {
            "n_neighbors": [3, 5, 9, 15, 25, 51],
            "weights": ["uniform", "distance"],
            "p": [1, 2],
        },
        "random_forest": {
            "max_depth": [6, 12, 20, None],
            "min_samples_leaf": [1, 2, 4],
        },
        "extra_trees": {"max_depth": [6, 12, 20, None], "min_samples_leaf": [1, 2, 4]},
        "hist_gradient": {
            "max_iter": [200, 400],
            "max_leaf_nodes": [15, 31],
            "learning_rate": [0.03, 0.1],
            "l2_regularization": [0.0, 1.0, 10.0],
        },
        "xgboost": {
            "n_estimators": [300, 600],
            "max_depth": [2, 3, 5],
            "learning_rate": [0.03, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
        },
        "catboost": {
            "iterations": [300, 600],
            "depth": [4, 6],
            "learning_rate": [0.03, 0.1],
            "l2_leaf_reg": [3.0, 10.0],
        },
    }
    rng = random.Random(SEED + int(digest(recipe)[:8], 16))
    candidates = {digest(recipe): copy.deepcopy(recipe)}
    while len(candidates) < count:
        item = copy.deepcopy(recipe)
        item["params"] = {
            key: rng.choice(values) for key, values in spaces[item["family"]].items()
        }
        item["log"] = rng.choice([False, True]) if item["group"] != "eye" else False
        item["ratio"] = rng.choice([False, True]) if item["group"] != "eye" else False
        if item["family"] in {"logistic", "svm", "knn"}:
            item["scaler"] = rng.choice(["standard", "robust"])
        candidates[digest(item)] = item
    return list(candidates.values())


def table_of(records):
    return pd.DataFrame(
        [
            {
                "id": r["id"],
                "window": r["window"],
                "stage": r["stage"],
                "model": r["recipe"].get("family", r["recipe"]["kind"]),
                "input": r["recipe"].get("group", "combined"),
                **r["validation"],
                "train_accuracy": r.get("training", {}).get("accuracy"),
                "seconds": r["seconds"],
            }
            for r in records
        ]
    ).sort_values(["window", "accuracy", "macro_f1"], ascending=[True, False, False])


def select_families(records, number=2):
    selected = {}
    for record in sorted(records, key=rank_key):
        family = record["recipe"].get("family")
        if family and family != "dummy" and family not in selected:
            selected[family] = record
    return list(selected.values())[:number]


def ensemble_candidate(store, window, children, weights, validation, stage):
    recipe = {
        "kind": "ensemble",
        "children": [child["recipe"] for child in children],
        "weights": weights,
    }
    key = f"w{window}_" + digest(recipe)[:20]
    previous = store.read(key)
    if previous is not None:
        return previous
    probabilities = combine_probabilities(
        [
            from_array_bytes(store.artifact(child["id"], "validation.npy"))
            for child in children
        ],
        weights,
    )
    record = {
        "id": key,
        "window": window,
        "recipe": recipe,
        "stage": stage,
        "complexity": len(children),
        "validation": metrics(validation["_y"], probabilities),
        "seconds": 0.0,
    }
    store.commit(key, record, {"validation.npy": array_bytes(probabilities)})
    return record

## Penyaringan algoritma dan sensor

Bandingkan Logistic Regression, SVM RBF, KNN, Random Forest, Extra Trees, HistGradientBoosting, XGBoost, dan CatBoost pada tiga kelompok sensor. Dummy menyediakan pembanding kelas konstan. Skor yang ditampilkan adalah validation.


In [ ]:
records_by_id: dict[str, Any] = {}


def remember(record):
    records_by_id[record["id"]] = record
    return record


families = [
    "logistic",
    "svm",
    "knn",
    "random_forest",
    "extra_trees",
    "hist_gradient",
    "xgboost",
    "catboost",
]
for window, view in views.items():
    train, validation = part_frame(view, "train"), part_frame(view, "validation")
    print(f"\nWindow {window}s: penyaringan algoritma")
    remember(
        evaluate_candidate(
            store, window, base_recipe("dummy", "fusion"), train, validation, "dummy"
        )
    )
    for family in families:
        for group in ("eye", "hrv", "fusion"):
            remember(
                evaluate_candidate(
                    store,
                    window,
                    base_recipe(family, group),
                    train,
                    validation,
                    "screening",
                )
            )
screening_table = table_of(list(records_by_id.values()))
display(screening_table.round(4))
publish_leaderboard(store, screening_table)

## Optimasi, fitur, balancing, dan ensemble

Dua keluarga terbaik setiap window menjalani pencarian 12 konfigurasi unik, termasuk kontrol. Pada kandidat teroptimasi, uji seleksi mutual information/PCA, class atau sample weight jika didukung, ROS, dan SMOTE. Seleksi dan balancing diuji terhadap kontrol yang sama, kemudian kombinasi terbaiknya dicoba.

Tambahan kandidat: Logistic Regression dengan interaksi derajat dua, hierarchy Low/Elevated lalu Medium/High, soft voting tiga keluarga terbaik, serta late fusion eye/HRV. Semua keputusan memakai validation, tidak memakai test. Banyak varian digabung di notebook ini; tidak ada nested LOPO atau pencarian tanpa batas.


In [ ]:
for window, view in views.items():
    train, validation = part_frame(view, "train"), part_frame(view, "validation")
    screening = [r for r in records_by_id.values() if r["window"] == window]
    leaders = select_families(screening, 2)
    print(f"\nWindow {window}s: optimasi dua keluarga terbaik")
    for leader in leaders:
        for recipe in tune_recipes(leader["recipe"], count=12):
            remember(
                evaluate_candidate(store, window, recipe, train, validation, "tuning")
            )

    tuned = select_families(
        [r for r in records_by_id.values() if r["window"] == window], 2
    )
    for control in tuned:
        selections, balances = [], []
        for selector in ("all", "mi8", "mi16", "pca"):
            recipe = copy.deepcopy(control["recipe"])
            recipe["selector"] = selector
            selections.append(
                remember(
                    evaluate_candidate(
                        store, window, recipe, train, validation, "feature_selection"
                    )
                )
            )
        methods = ["none", "ros", "smote"]
        if control["recipe"]["family"] != "knn":
            methods.append("weight")
        for balance in methods:
            recipe = copy.deepcopy(control["recipe"])
            recipe["balance"] = balance
            balances.append(
                remember(
                    evaluate_candidate(
                        store, window, recipe, train, validation, "balancing"
                    )
                )
            )
        recipe = copy.deepcopy(sorted(selections, key=rank_key)[0]["recipe"])
        recipe["balance"] = sorted(balances, key=rank_key)[0]["recipe"]["balance"]
        remember(
            evaluate_candidate(
                store, window, recipe, train, validation, "combined_selection_balance"
            )
        )

    for group in ("eye", "fusion"):
        for c_value in (0.1, 1.0, 10.0):
            recipe = base_recipe("logistic", group)
            recipe.update(polynomial=True, params={"C": c_value})
            remember(
                evaluate_candidate(
                    store, window, recipe, train, validation, "interactions"
                )
            )
    current = [
        r
        for r in records_by_id.values()
        if r["window"] == window and r["recipe"].get("family") != "dummy"
    ]
    leader = sorted(current, key=rank_key)[0]
    hierarchy = {"kind": "hierarchy", "child": leader["recipe"]}
    remember(
        evaluate_candidate(store, window, hierarchy, train, validation, "hierarchy")
    )

    singles = [
        r
        for r in records_by_id.values()
        if r["window"] == window and r["recipe"]["kind"] == "single"
    ]
    voting_children = select_families(singles, 3)
    remember(
        ensemble_candidate(
            store,
            window,
            voting_children,
            [1.0] * len(voting_children),
            validation,
            "soft_voting",
        )
    )
    eye = sorted([r for r in singles if r["recipe"]["group"] == "eye"], key=rank_key)[0]
    hrv = sorted([r for r in singles if r["recipe"]["group"] == "hrv"], key=rank_key)[0]
    for weight in (0.25, 0.5, 0.75):
        remember(
            ensemble_candidate(
                store,
                window,
                [eye, hrv],
                [weight, 1 - weight],
                validation,
                "late_fusion",
            )
        )
    publish_leaderboard(store, table_of(list(records_by_id.values())))

validation_table = table_of(list(records_by_id.values()))
display(validation_table.round(4))
plot_validation(validation_table)

## Refit dan evaluasi test

Pilih satu kandidat setiap window berdasarkan accuracy validation, kemudian Macro-F1, kompleksitas, dan ID sebagai urutan tetap. Pemenang utama antarwindow juga ditetapkan sebelum test dibuka. Finalist di-fit ulang pada training + validation; test tidak ikut fitting. Untuk neural, jumlah epoch refit berasal dari checkpoint terbaik training sebelumnya.

Model yang disimpan adalah model yang benar-benar dievaluasi pada test. Hasil meliputi accuracy, Macro-F1, balanced accuracy, precision/recall per kelas, dan confusion matrix. Target accuracy 75% ditampilkan sebagai tercapai/belum tercapai, bukan syarat untuk mengubah split. Hasil antarwindow mempunyai jumlah test yang berbeda. Jangan mengganti pemenang utama berdasarkan tabel test.


In [ ]:
finalists = []
for window in (15, 25, 45):
    eligible = [
        r
        for r in records_by_id.values()
        if r["window"] == window and r["recipe"].get("family") != "dummy"
    ]
    finalists.append(sorted(eligible, key=rank_key)[0])
print("Finalist dikunci dari validation sebelum evaluasi test:")
display(
    pd.DataFrame(
        [{"window": r["window"], "id": r["id"], **r["validation"]} for r in finalists]
    )
)
test_summary = final_evaluation(
    store,
    views,
    finalists,
    refit=lambda record, trainval: fit_tabular(
        record["recipe"], trainval, trainval["_y"].to_numpy()
    ),
    predict=predict_tabular,
)
display(test_summary.round(4))

## Membaca ulang model hasil evaluasi

Contoh ini memuat model utama dari run yang sama dan menampilkan probabilitas. Tidak ada training baru. Muat hanya artefak terpercaya yang dibuat tim. Definisi dalam notebook ini tetap diperlukan untuk menjalankan contoh; tidak ada file kode pendamping yang harus diunggah.


In [ ]:
# Contoh prediksi ulang model utama pada baris test yang sama, bukan training baru.
selected = store.read("selection")
final_key = "final_" + selected["primary_id"]
evaluated_model = load_bundle(store.artifact(final_key, "model.pkl"))
example = part_frame(views[selected["primary_window"]], "test").head(3)
probabilities = predict_tabular(evaluated_model, example)
display(pd.DataFrame(probabilities, columns=LABELS))